# Combine SQLite Code and Pytest Tests with Allure in One Notebook

in terminal run the following 
python -m venv .venv  
pip install pip install allure-pytest



This notebook:
1. Writes **app_code.py** (our SQLite logic).
2. Writes **test_app_code.py** (pytest + Allure tests).
3. Runs `pytest` from a notebook cell, saving Allure results to `allure-results/`.


'pytest notebook_test --maxfail=1 -v --alluredir=allure-results' via terminal

## 1. Create `app_code.py`
We define functions to:
- Create an **in-memory SQLite** database
- **Add** employees
- **Get** an employee by ID
- **Get** all employees in a department
- **Remove** an employee by ID

In [ ]:
%%writefile app_code.py
import sqlite3
from typing import List, Tuple, Optional

def create_in_memory_db() -> sqlite3.Connection:
    """
    Creates an in-memory SQLite database and returns the connection.
    Also initializes a simple schema: an 'employees' table.
    """
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()
    cursor.execute(
        """
        CREATE TABLE employees (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            department TEXT NOT NULL
        )
        """
    )
    conn.commit()
    return conn

def add_employee(conn: sqlite3.Connection, name: str, department: str) -> int:
    """
    Inserts a new employee into the 'employees' table.
    Returns the newly inserted employee's ID.
    """
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO employees (name, department) VALUES (?, ?)",
        (name, department)
    )
    conn.commit()
    return cursor.lastrowid

def get_employee_by_id(conn: sqlite3.Connection, emp_id: int) -> Optional[Tuple[int, str, str]]:
    """
    Fetches an employee by ID.
    Returns a tuple (id, name, department) if found, else None.
    """
    cursor = conn.cursor()
    cursor.execute(
        "SELECT id, name, department FROM employees WHERE id = ?",
        (emp_id,)
    )
    return cursor.fetchone()

def get_all_employees_in_department(conn: sqlite3.Connection, dept: str) -> List[Tuple[int, str, str]]:
    """
    Returns all employees in the given department as a list of (id, name, department).
    """
    cursor = conn.cursor()
    cursor.execute(
        "SELECT id, name, department FROM employees WHERE department = ?",
        (dept,)
    )
    return cursor.fetchall()

def remove_employee_by_id(conn: sqlite3.Connection, emp_id: int) -> bool:
    """
    Removes an employee by their ID.
    Returns True if an employee was removed, False otherwise.
    """
    cursor = conn.cursor()
    cursor.execute("DELETE FROM employees WHERE id = ?", (emp_id,))
    conn.commit()
    return cursor.rowcount > 0


## 2. Create `test_app_code.py`
We define **Pytest** tests with **Allure** annotations:

- Each test deals with a fresh, in-memory database for isolation.
- If you have [Allure CLI](https://docs.qameta.io/allure/#_get_started) installed, you can generate an HTML report.

In [ ]:
%%writefile test_app_code.py
import pytest
import allure
from app_code import (
    create_in_memory_db,
    add_employee,
    get_employee_by_id,
    get_all_employees_in_department,
    remove_employee_by_id
)

@allure.feature("Employee Management")
@allure.story("Add & Retrieve")
@allure.severity(allure.severity_level.NORMAL)
def test_add_employee_and_retrieve():
    """Tests that adding an employee and retrieving by ID works."""
    conn = create_in_memory_db()
    new_id = add_employee(conn, "Bob", "HR")
    row = get_employee_by_id(conn, new_id)
    assert row is not None, "Expected newly added employee to be found."  
    assert row[1] == "Bob", "Employee name should match."    
    assert row[2] == "HR",  "Employee department should match." 
    conn.close()

@allure.feature("Employee Management")
@allure.story("Retrieve")
@allure.severity(allure.severity_level.TRIVIAL)
def test_get_employee_by_id_not_found():
    """Tests that a non-existent ID returns None."""
    conn = create_in_memory_db()
    row = get_employee_by_id(conn, 9999)
    assert row is None, "Should get None if employee doesn't exist."    
    conn.close()

@allure.feature("Employee Management")
@allure.story("Query By Department")
@allure.severity(allure.severity_level.NORMAL)
def test_get_all_employees_in_department():
    """Tests retrieving all employees in a specific department."""
    conn = create_in_memory_db()
    add_employee(conn, "Carol", "Marketing")
    add_employee(conn, "Dave", "Marketing")
    add_employee(conn, "Eve", "Engineering")
    marketing_emps = get_all_employees_in_department(conn, "Marketing")
    assert len(marketing_emps) == 2, "Expected 2 employees in Marketing."  
    names = [emp[1] for emp in marketing_emps]
    assert set(names) == {"Carol", "Dave"}, "Should find Carol & Dave." 
    conn.close()

@allure.feature("Employee Management")
@allure.story("Removal")
@allure.severity(allure.severity_level.CRITICAL)
def test_remove_employee_by_id():
    """Tests removing an existing employee by ID."""
    conn = create_in_memory_db()
    emp_id = add_employee(conn, "Frank", "Engineering")
    removed = remove_employee_by_id(conn, emp_id)
    assert removed, "Expected removal of existing employee."  
    # Verify it's gone
    row = get_employee_by_id(conn, emp_id)
    assert row is None, "Employee should no longer exist."  
    conn.close()

@allure.feature("Employee Management")
@allure.story("Removal")
@allure.severity(allure.severity_level.NORMAL)
def test_remove_non_existent_employee():
    """Tests removing a non-existent employee returns False."""
    conn = create_in_memory_db()
    removed = remove_employee_by_id(conn, 9999)
    assert not removed, "Should return False when ID doesn't exist."  
    conn.close()


## 3. Run Pytest & Collect Allure Results
We use a **shell command** to run `pytest test_app_code.py` and direct Allure results to **`allure-results/`**.

> **Note**: If you get an error like `No module named 'app_code'`, confirm you’re in the **same directory** as these new `.py` files.

In [ ]:
!pytest test_app_code.py --alluredir=allure-results --maxfail=1 -v

If everything goes well, you’ll see the **test results** in the cell output, and a new folder **`allure-results/`** appears with the Allure JSON/XML files.

## 4. (Optional) Generate or Serve the Allure HTML Report
If you have the [Allure CLI](https://docs.qameta.io/allure/#_get_started) installed, you can:

- `allure generate allure-results --clean -o allure-report` to **generate** a static HTML.
- `allure serve allure-results` to **serve** it locally.

Below is an example cell to do so. Un-comment if you have Allure CLI installed and want to run it from the notebook.

In [ ]:
# !allure generate allure-results --clean -o allure-report
# !allure serve allure-results

## Conclusion

With this setup:
1. The **application code** is in `app_code.py`.
2. The **tests** (with **Allure** annotations) are in `test_app_code.py`.
3. We run **standard Pytest** from a cell (`!pytest ...`).
4. Allure results go to **`allure-results/`**.
5. (Optional) We generate/serve the **HTML report** with the Allure CLI.

Everything is driven from the notebook, **no** environment confusion about `code/` packages, and **no** `ipytest` needed.